# 04 · Silver: reclamações limpas e tipadas

Lê `mvp_reclamacoes.bronze.reclamacoes` e grava `mvp_reclamacoes.silver.reclamacoes`: **todos os registros de todos os setores**, limpos, tipados e com nomes em `snake_case`. O recorte bancário é regra de negócio e fica na Gold (*"at least one validated, non-aggregated representation of each record"*, [Medallion](https://docs.databricks.com/aws/en/lakehouse/medallion)).

As regras vêm do perfilamento (`03_perfilamento_bronze`):

| Regra | Evidência |
|---|---|
| `trim` em todo texto | Espaços sobrando em `Região` (`'N '`, `'S '`), `Nome Fantasia` e `Problema` (H4, H5, H2c) |
| `data_finalizacao` lida em `aaaa-mm-dd` e em `dd/mm/aaaa` | O arquivo de 2021-12 inteiro vem em `dd/mm/aaaa` (H8) |
| `tempo_resposta_dias` acima de 365 → nulo, com `tempo_resposta_invalido` | Valores contínuos até 80 dias, depois só 763 e 1.478.155 (H7b) |
| De-para de 6 nomes de `problema` | `–` trocado por `?` pela fonte a partir de 2021-12, 1 erro de digitação e 1 renomeação (H2b, H2c) |
| `nome_empresa` = grafia mais frequente entre as que só diferem em maiúsculas, espaços e acentos | Ex.: `'Banco BMG'` × `'Banco Bmg'` (H4) |
| `linha_repetida` marca linhas com cópia idêntica nas 21 colunas | 55.136 grupos, sempre no mesmo arquivo; sem ID não dá para saber se é duplicata (H9) |

Os valores originais de `Nome Fantasia` e `Problema` ficam em `nome_fantasia_original` e `problema_original`.

**Ordem:** as verificações da seção 3 rodam **antes** da gravação. Se alguma falhar, a tabela não é gravada.

## 1. Parâmetros e regras

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

BRONZE = "mvp_reclamacoes.bronze.reclamacoes"
SILVER = "mvp_reclamacoes.silver.reclamacoes"
SEGMENTOS_RECORTE = [
    "Bancos, Financeiras e Administradoras de Cartão",
    "Empresas de Pagamento Eletrônico",
]
LIMITE_TEMPO_RESPOSTA = 365  # dias; acima disso o valor vira nulo (H7b)

DE_PARA_PROBLEMA = {
    # "–" trocado por "?" na exportação da fonte a partir de 2021-12 (conferido nos bytes do arquivo original)
    "Negativação indevida sem contratação do serviço ? fraude bancária":
        "Negativação indevida sem contratação do serviço – fraude bancária",
    "Dados pessoais ou financeiros incorretos / desatualizados ? dificuldade de retificação":
        "Dados pessoais ou financeiros incorretos / desatualizados – dificuldade de retificação",
    "Recall ? descumprimento, dúvida": "Recall – descumprimento, dúvida",
    "Meia entrada ? recusa / falta de informação": "Meia entrada – recusa / falta de informação",
    # erro de digitação corrigido pela fonte em 2023-11
    "SAC - Dificuldadede de contato / acesso": "SAC - Dificuldade de contato / acesso",
    # renomeação em 2022-03, mesmo grupo (Contrato / Oferta)
    "Ligações indesejadas de telemarketing": "Ligações indesejadas de telemarketing (0303)",
}

COM_ACENTO = "áàâãäéèêëíìîïóòôõöúùûüçñ"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuucn"

# Colunas de texto copiadas com trim: nome na Bronze → nome na Silver
TEXTO = {
    "Região": "regiao", "UF": "uf", "Cidade": "cidade", "Sexo": "sexo", "Faixa Etária": "faixa_etaria",
    "Segmento de Mercado": "segmento_mercado", "Área": "area", "Assunto": "assunto",
    "Grupo Problema": "grupo_problema", "Como Comprou Contratou": "canal_contratacao",
    "Situação": "situacao", "Avaliação Reclamação": "avaliacao_reclamacao",
}

## 2. Transformações

In [ ]:
bronze = spark.table(BRONZE)
colunas_fonte = [c for c in bronze.columns if not c.startswith("_")]


def texto(coluna):
    return F.trim(F.col(coluna))


def sim_nao(coluna):
    return F.when(texto(coluna) == "S", True).when(texto(coluna) == "N", False)


# nome_empresa: grafia mais frequente dentro de cada chave normalizada (empate → ordem alfabética)
chave_empresa = F.translate(F.regexp_replace(F.lower(texto("Nome Fantasia")), r"\s+", " "), COM_ACENTO, SEM_ACENTO)
nomes = bronze.groupBy(chave_empresa.alias("chave_empresa"), texto("Nome Fantasia").alias("grafia")).count()
ordem = Window.partitionBy("chave_empresa").orderBy(F.desc("count"), F.asc("grafia"))
nome_canonico = (
    nomes.withColumn("posicao", F.row_number().over(ordem))
    .filter("posicao = 1")
    .select("chave_empresa", F.col("grafia").alias("nome_empresa"))
)

tempo = texto("Tempo Resposta").try_cast("int")
mapa_problema = F.create_map(*[F.lit(t) for par in DE_PARA_PROBLEMA.items() for t in par])

transformado = (
    bronze
    .withColumn("chave_empresa", chave_empresa)
    .join(F.broadcast(nome_canonico), "chave_empresa", "left")
    .withColumn("linha_repetida", F.count("*").over(Window.partitionBy(*colunas_fonte)) > 1)
    .select(
        # colunas brutas usadas só nas verificações (não vão para a Silver)
        *[F.col(c).alias(f"bruto_{i}") for i, c in enumerate(colunas_fonte)],
        *[texto(origem).alias(destino) for origem, destino in TEXTO.items()],
        F.coalesce(
            F.try_to_date(texto("Data Finalização"), "yyyy-MM-dd"),
            F.try_to_date(texto("Data Finalização"), "dd/MM/yyyy"),
        ).alias("data_finalizacao"),
        F.when(tempo <= LIMITE_TEMPO_RESPOSTA, tempo).alias("tempo_resposta_dias"),
        F.coalesce(tempo > LIMITE_TEMPO_RESPOSTA, F.lit(False)).alias("tempo_resposta_invalido"),
        F.col("Nome Fantasia").alias("nome_fantasia_original"),
        "nome_empresa",
        F.col("Problema").alias("problema_original"),
        F.coalesce(mapa_problema[texto("Problema")], texto("Problema")).alias("problema"),
        sim_nao("Procurou Empresa").alias("procurou_empresa"),
        sim_nao("Respondida").alias("respondida"),
        texto("Nota do Consumidor").try_cast("int").alias("nota_consumidor"),
        sim_nao("Interação com Judiciario").alias("interacao_judiciario"),
        F.try_to_date(texto("Último Complemento Consumidor"), "dd/MM/yyyy").alias("data_ultimo_complemento"),
        "linha_repetida",
        "_arquivo_origem",
        "_data_ingestao",
    )
)
bruto = {c: F.col(f"bruto_{i}") for i, c in enumerate(colunas_fonte)}

COLUNAS_SILVER = [
    "regiao", "uf", "cidade", "sexo", "faixa_etaria", "data_finalizacao", "tempo_resposta_dias",
    "tempo_resposta_invalido", "nome_fantasia_original", "nome_empresa", "segmento_mercado", "area", "assunto",
    "grupo_problema", "problema_original", "problema", "canal_contratacao", "procurou_empresa", "respondida",
    "situacao", "avaliacao_reclamacao", "nota_consumidor", "interacao_judiciario", "data_ultimo_complemento",
    "linha_repetida", "_arquivo_origem", "_data_ingestao",
]

## 3. Verificações (antes de gravar)

1. **Nenhuma conversão falha sem avisar:** nenhum valor preenchido na Bronze pode virar nulo na Silver. A única exceção são os tempos acima de 365 dias, que ficam marcados em `tempo_resposta_invalido`. Um valor fora de S/N nas colunas booleanas também cai aqui.
2. **`data_finalizacao` sempre dentro do mês do próprio arquivo.**
3. **Nenhum dos 6 nomes antigos da de-para sobra em `problema`.**
4. **`regiao` só em `CO`, `N`, `NE`, `S` ou `SE`; `nota_consumidor` só de 1 a 5.**
5. **No recorte bancário, cada `nome_empresa` tem um único segmento** (a padronização não misturou empresas).

In [ ]:
def virou_nulo(original, convertido):
    return F.sum((original.isNotNull() & convertido.isNull()).cast("int"))


mes_arquivo = F.regexp_extract("_arquivo_origem", r"(\d{4}-\d{2})", 1)
v = transformado.agg(
    F.count("*").alias("linhas"),
    virou_nulo(bruto["Data Finalização"], F.col("data_finalizacao")).alias("perda_data_finalizacao"),
    virou_nulo(bruto["Último Complemento Consumidor"], F.col("data_ultimo_complemento")).alias("perda_data_ultimo_complemento"),
    F.sum((bruto["Tempo Resposta"].isNotNull() & F.col("tempo_resposta_dias").isNull()
           & ~F.col("tempo_resposta_invalido")).cast("int")).alias("perda_tempo_resposta"),
    virou_nulo(bruto["Nota do Consumidor"], F.col("nota_consumidor")).alias("perda_nota"),
    virou_nulo(bruto["Respondida"], F.col("respondida")).alias("perda_respondida"),
    virou_nulo(bruto["Procurou Empresa"], F.col("procurou_empresa")).alias("perda_procurou_empresa"),
    virou_nulo(bruto["Interação com Judiciario"], F.col("interacao_judiciario")).alias("perda_interacao_judiciario"),
    F.sum(F.col("tempo_resposta_invalido").cast("int")).alias("tempos_invalidos"),
    F.sum((F.date_format("data_finalizacao", "yyyy-MM") != mes_arquivo).cast("int")).alias("datas_fora_do_mes"),
    F.sum(F.col("problema").isin(list(DE_PARA_PROBLEMA)).cast("int")).alias("de_para_restante"),
    F.sum((~F.coalesce(F.col("regiao").isin("CO", "N", "NE", "S", "SE"), F.lit(False))).cast("int")).alias("regiao_invalida"),
    F.sum((~F.coalesce(F.col("nota_consumidor").between(1, 5), F.lit(True))).cast("int")).alias("nota_fora_de_1_a_5"),
    # contagens para a seção 4 (mesma passada pelos dados)
    *[F.sum((bruto[c] != F.trim(bruto[c])).cast("int")).alias(f"trim_{c}")
      for c in list(TEXTO) + ["Nome Fantasia", "Problema"]],
    F.sum(F.trim(bruto["Problema"]).isin(list(DE_PARA_PROBLEMA)).cast("int")).alias("registros_com_de_para"),
    F.sum((F.col("nome_empresa") != F.trim(bruto["Nome Fantasia"])).cast("int")).alias("registros_com_nome_unificado"),
    F.sum(F.col("linha_repetida").cast("int")).alias("linhas_repetidas"),
).first().asDict()
CONTAGENS = [k for k in v if k.startswith("trim_")] + ["registros_com_de_para", "registros_com_nome_unificado", "linhas_repetidas"]

empresas_com_varios_segmentos = (
    transformado.filter(F.col("segmento_mercado").isin(SEGMENTOS_RECORTE))
    .groupBy("nome_empresa").agg(F.countDistinct("segmento_mercado").alias("segmentos"))
    .filter("segmentos > 1").count()
)
linhas_bronze = bronze.count()

for nome, valor in v.items():
    if nome not in CONTAGENS:
        print(f"{nome:<32} {valor:>12,}")
print(f"{'linhas_bronze':<32} {linhas_bronze:>12,}")
print(f"{'empresas_recorte_varios_segmentos':<32} {empresas_com_varios_segmentos:>12,}")

assert v["linhas"] == linhas_bronze, "a Silver precisa ter as mesmas linhas da Bronze"
for chave in [k for k in v if k.startswith("perda_")]:
    assert v[chave] == 0, f"{chave}: valor preenchido virou nulo na conversão"
assert v["datas_fora_do_mes"] == 0, "data_finalizacao fora do mês do arquivo"
assert v["de_para_restante"] == 0, "sobrou nome antigo da de-para em problema"
assert v["regiao_invalida"] == 0, "regiao fora de CO/N/NE/S/SE"
assert v["nota_fora_de_1_a_5"] == 0, "nota_consumidor fora de 1 a 5"
assert empresas_com_varios_segmentos == 0, "nome_empresa com mais de um segmento no recorte"
print("\nTodas as verificações passaram.")

## 4. O que cada limpeza mudou (para a seção de Qualidade)

Mostra quantos registros cada regra alterou (calculado na mesma passada da seção 3). É só registro, sem `assert`.

In [ ]:
for nome in CONTAGENS:
    print(f"{nome:<40} {v[nome]:>12,}")

grafias = (
    nomes.groupBy("chave_empresa").agg(F.count("*").alias("grafias"), F.sort_array(F.collect_list("grafia")).alias("variantes"))
    .filter("grafias > 1")
)
print(f"\nchaves de empresa com mais de uma grafia (Bronze inteira, depois do trim): {grafias.count()}")
for linha in grafias.orderBy("chave_empresa").limit(40).collect():
    print(f"  {linha['variantes']}")

## 5. Gravação e conferência

In [ ]:
transformado.select(*COLUNAS_SILVER).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER)

silver = spark.table(SILVER)
linhas_silver = silver.count()
print(f"linhas na Silver: {linhas_silver:,} | linhas na Bronze: {linhas_bronze:,}")
assert linhas_silver == linhas_bronze, "a tabela gravada tem número de linhas diferente da Bronze"
silver.printSchema()